# Stream Events to UC Volume

This notebook appends synthetic incremental logistics events to Unity Catalog volume landing zones.

**Purpose:**
Simulates real-time event ingestion by generating and appending new events to the raw data volumes. These events are then processed by the Spark Declarative Pipeline.

**Events Generated:**
- **Shipment Events** - Status updates, ETA changes
- **Incident Events** - New incidents, status updates
- **Capacity Events** - Utilization changes
- **Sensor Events** - Telemetry data (vibration, temperature, GPS)

**Schedule:**
This notebook is designed to run on a schedule (e.g., every 5 minutes) to simulate continuous data flow.

## Configuration

Load catalog and schema from job parameters, environment variables, or use defaults.

In [ ]:
from __future__ import annotations

import os
import random
import uuid
from datetime import datetime, date


def _get_config(name: str, default: str) -> str:
    """Get config from job parameters (widgets), env vars, or default."""
    try:
        return dbutils.widgets.get(name)  # type: ignore[name-defined]
    except Exception:
        return os.environ.get(f"DATABRICKS_{name.upper()}", default)


CATALOG = _get_config("catalog", "akrishn_fe_dsa")
SCHEMA = _get_config("schema", "logistics_control_center")
RAW_BASE = f"/Volumes/{CATALOG}/{SCHEMA}/raw_data"
now = datetime.now()

print(f"Catalog: {CATALOG}")
print(f"Schema: {SCHEMA}")
print(f"Raw data path: {RAW_BASE}")
print(f"Event time: {now}")

## Load Current State

Read existing data to generate realistic incremental events that reference actual entities.

In [ ]:
from pyspark.sql.functions import desc

lanes = [r.asDict() for r in spark.table(f"{CATALOG}.{SCHEMA}.lanes").select("id", "origin", "dest", "mode", "delayMinutes").collect()]
shipments = [r.asDict() for r in spark.table(f"{CATALOG}.{SCHEMA}.shipments").select("trackingId", "laneId", "status").orderBy(desc("currentETA")).limit(40).collect()]
incidents = [r.asDict() for r in spark.table(f"{CATALOG}.{SCHEMA}.incidents").select("id", "laneId", "type", "active").limit(20).collect()]
capacity = [r.asDict() for r in spark.table(f"{CATALOG}.{SCHEMA}.capacity_lanes").select("id", "availableCapacity", "utilizationPct").collect()]

print(f"Loaded: {len(lanes)} lanes, {len(shipments)} recent shipments, {len(incidents)} incidents, {len(capacity)} capacity records")

## Generate Shipment Events

Create status update events for recent shipments. Most remain in transit; a small percentage are marked as delivered.

In [ ]:
shipment_events: list[dict] = []

for s in shipments[: min(10, len(shipments))]:
    shipment_events.append(
        {
            "event_id": str(uuid.uuid4()),
            "trackingId": s["trackingId"],
            "laneId": s["laneId"],
            "event_time": now,
            "status": "delivered" if random.random() < 0.08 else "in_transit",
            "eta_delta_minutes": random.randint(-20, 45),
            "event_date": date.today(),
        }
    )

print(f"Generated {len(shipment_events)} shipment events")

## Generate Incident Events

Create status updates for existing incidents (most remain active, some resolve). Occasionally generate new incidents.

In [ ]:
incident_events: list[dict] = []

# Update existing incidents
for i in incidents[: min(4, len(incidents))]:
    incident_events.append(
        {
            "event_id": str(uuid.uuid4()),
            "incident_id": i["id"],
            "laneId": i["laneId"],
            "event_time": now,
            "type": i["type"],
            "active": random.random() >= 0.2,  # 80% stay active
            "impact_minutes": random.randint(20, 180),
            "event_date": date.today(),
        }
    )

# Occasionally generate new incident (35% chance)
if random.random() < 0.35 and lanes:
    lane = random.choice(lanes)
    incident_events.append(
        {
            "event_id": str(uuid.uuid4()),
            "incident_id": f"INC-{uuid.uuid4().hex[:8].upper()}",
            "laneId": lane["id"],
            "event_time": now,
            "type": random.choice(["weather", "equipment_issue", "air_traffic_control", "highway_delay"]),
            "active": True,
            "impact_minutes": random.randint(25, 160),
            "event_date": date.today(),
        }
    )

print(f"Generated {len(incident_events)} incident events")

## Generate Capacity Events

Update capacity utilization for lanes with small random fluctuations.

In [ ]:
capacity_events: list[dict] = []

for c in capacity[: min(8, len(capacity))]:
    jitter = random.uniform(-0.06, 0.06)
    new_util = max(0.4, min(0.99, float(c["utilizationPct"]) + jitter))
    new_avail = max(0, int(float(c["availableCapacity"]) * random.uniform(0.9, 1.1)))
    capacity_events.append(
        {
            "event_id": str(uuid.uuid4()),
            "laneId": c["id"],
            "event_time": now,
            "availableCapacity": new_avail,
            "utilizationPct": round(new_util, 4),
            "event_date": date.today(),
        }
    )

print(f"Generated {len(capacity_events)} capacity events")

## Generate Sensor Events

Simulate IoT sensor telemetry data: vibration, temperature, GPS delays, and weather risk scores.

In [ ]:
sensor_events: list[dict] = []

for lane in lanes[: min(8, len(lanes))]:
    sensor_events.append(
        {
            "event_id": str(uuid.uuid4()),
            "lane_id": lane["id"],
            "center_id": lane["origin"],
            "event_time": now,
            "vibration": round(random.uniform(0.8, 4.1), 4),
            "temp_c": round(random.uniform(6.0, 35.0), 2),
            "gps_delay_minutes": max(0, int(lane.get("delayMinutes") or 0) + random.randint(-15, 20)),
            "weather_risk": round(random.uniform(0.01, 0.95), 4),
            "event_date": date.today(),
        }
    )

print(f"Generated {len(sensor_events)} sensor events")

## Write Events to UC Volume

Append the generated events to the raw data landing zones in Unity Catalog volumes. The pipeline will process these on its next run.

In [ ]:
if shipment_events:
    spark.createDataFrame(shipment_events).coalesce(1).write.mode("append").json(f"{RAW_BASE}/shipment_events")
    print(f"✓ Wrote {len(shipment_events)} shipment events")

if incident_events:
    spark.createDataFrame(incident_events).coalesce(1).write.mode("append").json(f"{RAW_BASE}/incident_events")
    print(f"✓ Wrote {len(incident_events)} incident events")

if capacity_events:
    spark.createDataFrame(capacity_events).coalesce(1).write.mode("append").json(f"{RAW_BASE}/capacity_events")
    print(f"✓ Wrote {len(capacity_events)} capacity events")

if sensor_events:
    spark.createDataFrame(sensor_events).coalesce(1).write.mode("append").parquet(f"{RAW_BASE}/sensor_events")
    print(f"✓ Wrote {len(sensor_events)} sensor events")

## Summary

Display totals for the incremental event batch.

In [ ]:
print("\n" + "="*50)
print("Stream events to volume complete")
print("="*50)
print(
    f"Wrote {len(shipment_events)} shipment, {len(incident_events)} incident, "
    f"{len(capacity_events)} capacity, {len(sensor_events)} sensor events"
)